# OpenWakeWord: Custom Human Voice Training
This notebook is optimized for training a wake word model primarily on your **real voice recordings**.
It still generates a small amount of synthetic data to create "hard negatives" (words that sound like your wake word but aren't) to prevent false activations, but the vast majority of the training will focus on your actual voice!

In [4]:
import requests

try:
    response = requests.get("https://www.google.com", timeout=5)
    print("✅ Internet is ON")
except requests.ConnectionError:
    print("❌ Internet is OFF")
except Exception as e:
    print("Error:", e)

✅ Internet is ON


### Step 1: Install Dependencies
Run this cell to install OpenWakeWord and download the required background noise datasets.

In [24]:
!git clone https://github.com/dscripka/openWakeWord.git
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad
%cd openWakeWord
!pip install -e .
!pip install tensorflow
!pip install pronouncing
!pip install audiomentations
!pip install torch-audiomentations
!pip install speechbrain
!pip install mutagen
!pip install acoustics
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

fatal: could not create work tree dir 'openWakeWord': No space left on device
fatal: could not create work tree dir 'piper-sample-generator': No space left on device
piper-sample-generator/models/en_US-libritts_r-medium.pt: No such file or directory
ERROR: Could not find a version that satisfies the requirement piper-phonemize (from versions: none)
ERROR: No matching distribution found for piper-phonemize
[Errno 2] No such file or directory: 'openWakeWord'
/kaggle/working/openWakeWord/openWakeWord/openWakeWord/openWakeWord
Obtaining file:///kaggle/working/openWakeWord/openWakeWord/openWakeWord/openWakeWord
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build editabl

### Step 2: Upload Your Voice Clips
1. Record 50-100 clips of yourself saying your wake word (16kHz, 16-bit, Mono `.wav` format).
2. Put them in a `.zip` file named `my_positive_clips.zip`.
3. Upload that `.zip` file to the Colab sidebar.
4. Run the cell below to extract them!

In [10]:
!find /kaggle/input/datasets/aashutosh142240/my-voice

/kaggle/input/datasets/aashutosh142240/my-voice
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (5).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (3).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (20).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (11).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (17).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (6).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (2).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (18).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (8).wav
/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips/Recording (25).wav
/kaggle/input/datasets/aashutosh142240/my

In [6]:
#Only For Google Colab
!unzip /content/my_positive_clips.zip -d /content/
print("Voice clips extracted to /content/my_positive_clips/")

unzip:  cannot find or open /content/my_positive_clips.zip, /content/my_positive_clips.zip.zip or /content/my_positive_clips.zip.ZIP.
Voice clips extracted to /content/my_positive_clips/


### Step 3: Configure Your Wake Word
Change the `target_phrase` below to the word(s) you recorded yourself saying.

In [14]:
import yaml

config = {
    "target_phrase": ["ultron"], # Change this to your wake word
    "model_name": "custom_voice_model",
    "n_samples": 100, # Keep this low so it focuses on your real voice, but generates some synthetic variations
    "n_samples_val": 100,
    "steps": 10000,
    "target_accuracy": 0.6,
    "target_recall": 0.25,
    "background_paths": ['./audioset_16k', './fma'],
    "false_positive_validation_data_path": "validation_set_features.npy",
    "feature_data_files": {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print("Configuration saved!")

Configuration saved!


### Step 4: Generate Synthetic Negatives & Phonetic Variations
This runs quickly because we lowered the `n_samples`. It generates words that sound similar to your wake word to teach the model what *not* to trigger on.

In [22]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --generate_clips
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --augment_clips

Traceback (most recent call last):
  File "/kaggle/working/openWakeWord/openWakeWord/openWakeWord/openwakeword/train.py", line 645, in <module>
    sys.path.insert(0, os.path.abspath(config["piper_sample_generator_path"]))
                                       ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyError: 'piper_sample_generator_path'
Traceback (most recent call last):
  File "/kaggle/working/openWakeWord/openWakeWord/openWakeWord/openwakeword/train.py", line 645, in <module>
    sys.path.insert(0, os.path.abspath(config["piper_sample_generator_path"]))
                                       ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyError: 'piper_sample_generator_path'


### Step 5: Inject Your Real Human Voice Data
This cell converts your `.wav` files into OpenWakeWord features and seamlessly merges them into the training dataset.

In [ ]:
import os
import glob
import numpy as np
from openwakeword.utils import AudioFeatures

#custom_clips_dir = "/content/my_positive_clips" # uncomment for google colab
custom_clips_dir = "/kaggle/input/datasets/aashutosh142240/my-voice/my_positive_clips" #uncomment for kaggle notebook
if os.path.exists(custom_clips_dir):
    positive_clips = glob.glob(os.path.join(custom_clips_dir, "*.wav"))
    if len(positive_clips) > 0:
        print(f"Found {len(positive_clips)} custom positive clips. Extracting features...")
        F = AudioFeatures(device="cpu")
        custom_features = F.embed_clips(positive_clips, batch_size=16)
        
        if isinstance(custom_features, dict):
            custom_features = list(custom_features.values())[0]
        if isinstance(custom_features, list):
            custom_features = np.vstack(custom_features)
            
        model_name = config["model_name"]
        output_dir = os.path.join(config.get("output_dir", "my_custom_model"), model_name)
        train_path = os.path.join(output_dir, "positive_features_train.npy")
        val_path = os.path.join(output_dir, "positive_features_val.npy")
        
        if os.path.exists(train_path):
            existing_train = np.load(train_path)
            # We repeat the custom human features multiple times so the model focuses heavily on them.
            # MULTIPLIER: Set this to 4 (or higher) to repeat your 26 recordings multiple times.
            weight_multiplier = 4
            weighted_custom = np.tile(custom_features, (weight_multiplier, 1))
            new_train = np.vstack([existing_train, weighted_custom])
            np.save(train_path, new_train)
            print(f"Appended weighted real clips to training features. New total: {new_train.shape[0]}")
            
        if os.path.exists(val_path):
            existing_val = np.load(val_path)
            val_features = custom_features[:max(1, len(custom_features)//5)]
            new_val = np.vstack([existing_val, val_features])
            np.save(val_path, new_val)
            print(f"Appended real clips to validation features. New total: {new_val.shape[0]}")
    else:
        print(f"Directory {custom_clips_dir} exists, but no .wav files found.")
else:
    print(f"No custom real clips found at {custom_clips_dir}. Skipping...")

### Step 6: Train the Model!
Train the final neural network. Once complete, download the `.onnx` file from `/content/openWakeWord/my_custom_model/custom_voice_model/`.

In [ ]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --train_model